# 03 — Silver → Gold (Delta Lake)

Lê as tabelas **Delta** da camada **Silver** no MinIO, constrói um modelo dimensional
**estrela** e persiste as tabelas analíticas como **Delta Lake** na camada **Gold**.

### Modelo dimensional
| Tabela | Tipo | Grain / Descrição |
|---|---|---|
| `dim_date` | Dimensão | Calendário gerado a partir do intervalo de pedidos |
| `dim_customer` | Dimensão | Clientes com lat/lng médio por CEP |
| `dim_seller` | Dimensão | Vendedores com lat/lng médio por CEP |
| `dim_product` | Dimensão | Produtos com categoria denormalizada (star) |
| `dim_agent` | Dimensão | Agentes de suporte |
| `fact_orders` | Fato | Order-item — preço, frete, entrega, flags is_delivered/is_pending |
| `fact_reviews` | Fato | Avaliação — review_score, has_comment |
| `fact_support_tickets` | Fato | Ticket — is_solved, sla_met, message_count |

### KPIs e métricas suportados
| | Fonte |
|---|---|
| **Total revenue** | `fact_orders` WHERE `is_delivered = true` → SUM(`total_value`) |
| **Avg delivery time** | `fact_orders` WHERE `is_delivered = true` → AVG(`delivery_days`) |
| **Pending orders** | `fact_orders` WHERE `is_pending = true` → COUNT DISTINCT `order_id` |
| **Avg review score** | `fact_reviews` → AVG(`review_score`) |
| **Rating score over time** | `fact_reviews` GROUP BY `date_id` → AVG(`review_score`) |
| **Solved tickets over time** | `fact_support_tickets` WHERE `is_solved = true` GROUP BY `date_id` |

### Papermill
```bash
cd notebooks && uv run papermill 03_silver_to_gold.ipynb /dev/null
```

In [1]:
from __future__ import annotations

import json
import logging
import os
import re
from pathlib import Path
from typing import Any

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import pyspark.sql.types as T
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

In [2]:
# ──────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("silver_to_gold")

In [3]:
# ──────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent


def _load_env() -> None:
    """Carrega variáveis do arquivo .env (formato KEY=VALUE) sem depender de python-dotenv."""
    env_path = PROJECT_ROOT / ".env"
    if not env_path.exists():
        log.warning(".env não encontrado em %s – usando variáveis do sistema", env_path)
        return
    with open(env_path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


_load_env()

2026-06-22 21:47:03  WARNING   .env não encontrado em /mnt/c/source/.env – usando variáveis do sistema


In [4]:
# ──────────────────────────────────────────────
# Parâmetros (Papermill)
# ──────────────────────────────────────────────
# Esta célula é marcada com a tag 'parameters'.
# O Papermill injeta uma nova célula logo abaixo com os valores recebidos.
silver_bucket: str = os.getenv("MINIO_BUCKET_SILVER", "silver")
gold_bucket: str = os.getenv("MINIO_BUCKET_GOLD", "gold")
write_mode: str = "overwrite"

In [5]:
# Parameters
silver_bucket = "silver"
gold_bucket = "gold"


In [6]:
# ──────────────────────────────────────────────
# SparkSession
# ──────────────────────────────────────────────
def _create_spark_session() -> SparkSession:
    """
    Cria uma SparkSession local configurada para:
      - Acessar MinIO via protocolo S3A (hadoop-aws)
      - Usar Delta Lake como formato de tabela
    """
    endpoint   = os.getenv("MINIO_ENDPOINT",   "http://localhost:9000")
    access_key = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
    secret_key = os.getenv("MINIO_SECRET_KEY", "minioadmin")

    builder = (
        SparkSession.builder
        .appName("SilverToGold")
        .master("local[*]")
        # ── Delta Lake ──
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        # ── Hadoop / S3A para MinIO ──
        .config("spark.hadoop.fs.s3a.endpoint", endpoint)
        .config("spark.hadoop.fs.s3a.access.key", access_key)
        .config("spark.hadoop.fs.s3a.secret.key", secret_key)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
        )
        # ── Performance ──
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.driver.memory", "2g")
    )

    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    return spark


spark = _create_spark_session()

26/06/22 21:47:12 WARN Utils: Your hostname, PC-Octavio resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/22 21:47:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/source/projeto-final-ed/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/octavio/.ivy2/cache
The jars for the packages stored in: /home/octavio/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aa2445e4-7d58-420a-bb06-7f814a55bd8b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


:: resolution report :: resolve 193ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-aa2445e4-7d58-420a-bb06-7f814a55bd8b
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/5ms)


26/06/22 21:47:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [7]:
# ──────────────────────────────────────────────
# Leitura das tabelas Silver
# ──────────────────────────────────────────────
_META_COLS = {"_ingestion_timestamp", "_source_file", "_silver_timestamp", "_silver_source"}


def read_silver(table_name: str) -> DataFrame:
    """Lê uma tabela Delta do Silver, excluindo colunas de metadados de pipeline."""
    path = f"s3a://{silver_bucket}/{table_name}"
    df = spark.read.format("delta").load(path)
    drop_cols = [c for c in df.columns if c in _META_COLS]
    return df.drop(*drop_cols)


log.info("Carregando tabelas Silver...")

orders      = read_silver("olist_orders_dataset")
order_items = read_silver("olist_order_items_dataset")
customers   = read_silver("olist_customers_dataset")
sellers     = read_silver("olist_sellers_dataset")
products    = read_silver("olist_products_dataset")
category_tr = read_silver("product_category_name_translation")
geolocation = read_silver("olist_geolocation_dataset")
reviews     = read_silver("olist_order_reviews_dataset")
agents      = read_silver("agents")
tickets     = read_silver("support_tickets")
messages    = read_silver("support_ticket_messages")

log.info("Tabelas Silver carregadas.")

2026-06-22 21:47:16  INFO      Carregando tabelas Silver...


26/06/22 21:47:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


2026-06-22 21:47:28  INFO      Tabelas Silver carregadas.


In [8]:
# ──────────────────────────────────────────────
# dim_date
# Gerado a partir do intervalo de order_purchase_timestamp.
# ──────────────────────────────────────────────
min_ts, max_ts = (
    orders
    .select(
        F.min("order_purchase_timestamp").alias("min_ts"),
        F.max("order_purchase_timestamp").alias("max_ts"),
    )
    .collect()[0]
)

n_days = (max_ts.date() - min_ts.date()).days + 1
start_str = str(min_ts.date())

dim_date = (
    spark.range(n_days)
    .select(F.expr(f"date_add(to_date('{start_str}'), CAST(id AS INT))").alias("full_date"))
    .select(
        (F.year("full_date") * 10000 + F.month("full_date") * 100 + F.dayofmonth("full_date"))
            .cast(T.IntegerType()).alias("date_id"),
        F.col("full_date"),
        F.year("full_date").alias("year"),
        F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofmonth("full_date").alias("day"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        (F.dayofweek("full_date").isin([1, 7])).alias("is_weekend"),
    )
    .withColumn("date_sk", F.row_number().over(Window.orderBy("date_id")))
    .select(
        "date_sk", "date_id", "full_date", "year", "quarter", "month",
        "month_name", "day", "week_of_year", "day_of_week", "day_name", "is_weekend",
    )
)

log.info("dim_date: %d linhas", dim_date.count())

26/06/22 21:47:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


2026-06-22 21:47:34  INFO      dim_date: 774 linhas


In [9]:
# ──────────────────────────────────────────────
# dim_customer
# Enriquecido com lat/lng médio do CEP via geolocation.
# ──────────────────────────────────────────────

# Geolocation já está como IntegerType em Silver; reutilizado também em dim_seller.
geo_agg = geolocation.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("avg_lat"),
    F.avg("geolocation_lng").alias("avg_lng"),
)

dim_customer = (
    customers
    .join(
        geo_agg,
        customers["customer_zip_code_prefix"].cast(T.IntegerType())
        == geo_agg["geolocation_zip_code_prefix"],
        "left",
    )
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        F.round("avg_lat", 6).alias("avg_lat"),
        F.round("avg_lng", 6).alias("avg_lng"),
    )
    .withColumn("customer_sk", F.row_number().over(Window.orderBy("customer_id")))
    .select(
        "customer_sk", "customer_id", "customer_unique_id",
        "customer_zip_code_prefix", "customer_city", "customer_state",
        "avg_lat", "avg_lng",
    )
)

log.info("dim_customer: %d linhas", dim_customer.count())

2026-06-22 21:47:35  INFO      dim_customer: 99441 linhas


In [10]:
# ──────────────────────────────────────────────
# dim_seller
# Enriquecido com lat/lng médio do CEP (mesmo geo_agg de dim_customer).
# ──────────────────────────────────────────────
dim_seller = (
    sellers
    .join(
        geo_agg,
        sellers["seller_zip_code_prefix"].cast(T.IntegerType())
        == geo_agg["geolocation_zip_code_prefix"],
        "left",
    )
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        F.round("avg_lat", 6).alias("avg_lat"),
        F.round("avg_lng", 6).alias("avg_lng"),
    )
    .withColumn("seller_sk", F.row_number().over(Window.orderBy("seller_id")))
    .select(
        "seller_sk", "seller_id", "seller_zip_code_prefix",
        "seller_city", "seller_state", "avg_lat", "avg_lng",
    )
)

log.info("dim_seller: %d linhas", dim_seller.count())

2026-06-22 21:47:36  INFO      dim_seller: 3095 linhas


In [11]:
# ──────────────────────────────────────────────
# dim_product  (star — categoria denormalizada)
# product_category_name_english unido diretamente,
# sem tabela de categoria separada.
# ──────────────────────────────────────────────
dim_product = (
    products
    .join(category_tr, "product_category_name", "left")
    .select(
        "product_id",
        "product_category_name",
        F.coalesce(
            F.col("product_category_name_english"), F.lit("unknown")
        ).alias("product_category_name_english"),
        # Nomes com typo preservados do Silver para manter consistência.
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    )
    .withColumn("product_sk", F.row_number().over(Window.orderBy("product_id")))
    .select(
        "product_sk", "product_id", "product_category_name", "product_category_name_english",
        "product_name_lenght", "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    )
)

log.info("dim_product: %d linhas", dim_product.count())

2026-06-22 21:47:38  INFO      dim_product: 32951 linhas


In [12]:
# ──────────────────────────────────────────────
# dim_agent
# ──────────────────────────────────────────────
dim_agent = (
    agents
    .select("agent_id", "alias", "team")
    .withColumn("agent_sk", F.row_number().over(Window.orderBy("agent_id")))
    .select("agent_sk", "agent_id", "alias", "team")
)

log.info("dim_agent: %d linhas", dim_agent.count())

2026-06-22 21:47:39  INFO      dim_agent: 50 linhas


In [13]:
# ──────────────────────────────────────────────
# fact_orders  (grain: order_item)
#
# KPIs:
#   total_revenue   → SUM(total_value)  WHERE is_delivered = true
#   avg_delivery    → AVG(delivery_days) WHERE is_delivered = true
#   pending_orders  → COUNT DISTINCT order_id WHERE is_pending = true
# ──────────────────────────────────────────────
_orders_slim = orders.select(
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
)

fact_orders = (
    order_items
    .join(_orders_slim, "order_id", "inner")
    .withColumn(
        "date_id",
        (
            F.year("order_purchase_timestamp") * 10000
            + F.month("order_purchase_timestamp") * 100
            + F.dayofmonth("order_purchase_timestamp")
        ).cast(T.IntegerType()),
    )
    .withColumn("total_value", F.col("price") + F.col("freight_value"))
    .withColumn("is_delivered", F.col("order_status") == "delivered")
    .withColumn(
        "is_pending",
        (~F.col("order_status").isin("canceled", "unavailable"))
        & F.col("order_delivered_customer_date").isNull(),
    )
    .withColumn(
        "delivery_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff("order_delivered_customer_date", "order_purchase_timestamp"),
        ).otherwise(F.lit(None).cast(T.IntegerType())),
    )
    .select(
        "order_id",
        "order_item_id",
        "customer_id",
        "product_id",
        "seller_id",
        "date_id",
        "order_status",
        "is_delivered",
        "is_pending",
        F.col("price").cast(T.DoubleType()),
        F.col("freight_value").cast(T.DoubleType()),
        F.col("total_value").cast(T.DoubleType()),
        "delivery_days",
    )
)

log.info("fact_orders: %d linhas", fact_orders.count())

2026-06-22 21:47:41  INFO      fact_orders: 80223 linhas


In [14]:
# ──────────────────────────────────────────────
# fact_reviews  (grain: review)
#
# KPI:    avg_review_score → AVG(review_score)
# Métrica: rating score over time → AVG(review_score) GROUP BY date_id
# ──────────────────────────────────────────────
_orders_for_reviews = orders.select("order_id", "customer_id")

fact_reviews = (
    reviews
    .join(_orders_for_reviews, "order_id", "left")
    .withColumn(
        "date_id",
        (
            F.year("review_creation_date") * 10000
            + F.month("review_creation_date") * 100
            + F.dayofmonth("review_creation_date")
        ).cast(T.IntegerType()),
    )
    .withColumn(
        "has_comment",
        F.col("review_comment_message").isNotNull()
        & (F.trim(F.col("review_comment_message")) != ""),
    )
    .select(
        "review_id",
        "order_id",
        "customer_id",
        "date_id",
        "review_score",
        "has_comment",
    )
)

log.info("fact_reviews: %d linhas", fact_reviews.count())

2026-06-22 21:47:42  INFO      fact_reviews: 2872 linhas


In [15]:
# ──────────────────────────────────────────────
# fact_support_tickets  (grain: ticket)
#
# Métrica: solved tickets over time → COUNT(*) WHERE is_solved = true GROUP BY date_id
# ──────────────────────────────────────────────
_msg_count = messages.groupBy("ticket_id").agg(
    F.count("message_id").alias("message_count")
)

fact_support_tickets = (
    tickets
    .join(_msg_count, "ticket_id", "left")
    .withColumn(
        "date_id",
        (
            F.year("opened_at") * 10000
            + F.month("opened_at") * 100
            + F.dayofmonth("opened_at")
        ).cast(T.IntegerType()),
    )
    .withColumn("is_solved", F.col("status").isin("closed", "resolved"))
    .withColumn(
        "sla_met",
        F.when(
            F.col("resolution_hours").isNotNull() & F.col("sla_target_hours").isNotNull(),
            F.col("resolution_hours") <= F.col("sla_target_hours"),
        ).otherwise(F.lit(None).cast(T.BooleanType())),
    )
    .withColumn("message_count", F.coalesce(F.col("message_count"), F.lit(0)))
    .select(
        "ticket_id",
        "order_id",
        "customer_id",
        "agent_id",
        "date_id",
        "channel",
        "issue_type",
        "priority",
        "status",
        "is_solved",
        "first_response_minutes",
        "sla_target_hours",
        "resolution_hours",
        "sla_met",
        "requires_seller_action",
        "message_count",
    )
)

log.info("fact_support_tickets: %d linhas", fact_support_tickets.count())

2026-06-22 21:47:43  INFO      fact_support_tickets: 12000 linhas


In [16]:
# ──────────────────────────────────────────────
# Escrita — Gold (Delta Lake)
# ──────────────────────────────────────────────
GOLD_TABLES: dict[str, DataFrame] = {
    "dim_date":             dim_date,
    "dim_customer":         dim_customer,
    "dim_seller":           dim_seller,
    "dim_product":          dim_product,
    "dim_agent":            dim_agent,
    "fact_orders":          fact_orders,
    "fact_reviews":         fact_reviews,
    "fact_support_tickets": fact_support_tickets,
}

write_log: list[dict[str, Any]] = []

log.info("=" * 60)
log.info("Persistindo tabelas Gold")
log.info("=" * 60)
log.info("  Gold bucket: s3a://%s/", gold_bucket)

for table_name, df in GOLD_TABLES.items():
    target_path = f"s3a://{gold_bucket}/{table_name}"
    try:
        row_count = df.count()
        (
            df
            .withColumn("_gold_timestamp", F.current_timestamp())
            .withColumn("_gold_source", F.lit(silver_bucket))
            .write
            .format("delta")
            .mode(write_mode)
            .option("overwriteSchema", "true")
            .save(target_path)
        )
        log.info("  ✔ %-28s | %d linhas gravadas", table_name, row_count)
        write_log.append({"table": table_name, "rows": row_count, "status": "ok"})
    except Exception as exc:
        log.error("  ✘ %-28s | ERRO: %s", table_name, exc)
        write_log.append({"table": table_name, "status": "error", "error": str(exc)})

print(json.dumps(write_log, ensure_ascii=False, indent=2))

2026-06-22 21:47:43  INFO      ============================================================


2026-06-22 21:47:43  INFO      Persistindo tabelas Gold


2026-06-22 21:47:43  INFO      ============================================================


2026-06-22 21:47:43  INFO        Gold bucket: s3a://gold/


26/06/22 21:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/22 21:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/22 21:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/22 21:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/22 21:47:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


...

2026-06-22 21:47:55  INFO        ✔ dim_agent                    | 50 linhas gravadas


2026-06-22 21:48:00  INFO        ✔ fact_orders                  | 80223 linhas gravadas


2026-06-22 21:48:03  INFO        ✔ fact_reviews                 | 2872 linhas gravadas


2026-06-22 21:48:06  INFO        ✔ fact_support_tickets         | 12000 linhas gravadas


[
  {
    "table": "dim_date",
    "rows": 774,
    "status": "ok"
  },
  {
    "table": "dim_customer",
    "rows": 99441,
    "status": "ok"
  },
  {
    "table": "dim_seller",
    "rows": 3095,
    "status": "ok"
  },
  {
    "table": "dim_product",
    "rows": 32951,
    "status": "ok"
  },
  {
    "table": "dim_agent",
    "rows": 50,
    "status": "ok"
  },
  {
    "table": "fact_orders",
    "rows": 80223,
    "status": "ok"
  },
  {
    "table": "fact_reviews",
    "rows": 2872,
    "status": "ok"
  },
  {
    "table": "fact_support_tickets",
    "rows": 12000,
    "status": "ok"
  }
]


In [17]:
# ──────────────────────────────────────────────
# Verificação das tabelas Gold (Delta history)
# ──────────────────────────────────────────────
log.info("=" * 60)
log.info("VERIFICAÇÃO DAS TABELAS DELTA GOLD")
log.info("=" * 60)

for table_name in GOLD_TABLES:
    target_path = f"s3a://{gold_bucket}/{table_name}"
    try:
        dt = DeltaTable.forPath(spark, target_path)
        history = dt.history(1).select("version", "timestamp", "operation").collect()
        row = history[0]
        log.info(
            "  ✔ %-28s | v%s | %s | %s",
            table_name,
            row["version"],
            row["timestamp"],
            row["operation"],
        )
    except Exception as exc:
        log.error("  ✘ %-28s | ERRO: %s", table_name, exc)

2026-06-22 21:48:06  INFO      ============================================================


2026-06-22 21:48:06  INFO      VERIFICAÇÃO DAS TABELAS DELTA GOLD


2026-06-22 21:48:06  INFO      ============================================================


2026-06-22 21:48:07  INFO        ✔ dim_date                     | v0 | 2026-06-22 21:47:45 | WRITE


2026-06-22 21:48:07  INFO        ✔ dim_customer                 | v0 | 2026-06-22 21:47:49 | WRITE


...

2026-06-22 21:48:08  INFO        ✔ dim_product                  | v0 | 2026-06-22 21:47:53 | WRITE


2026-06-22 21:48:09  INFO        ✔ dim_agent                    | v0 | 2026-06-22 21:47:55 | WRITE


2026-06-22 21:48:09  INFO        ✔ fact_orders                  | v0 | 2026-06-22 21:48:00 | WRITE


2026-06-22 21:48:09  INFO        ✔ fact_reviews                 | v0 | 2026-06-22 21:48:03 | WRITE


2026-06-22 21:48:10  INFO        ✔ fact_support_tickets         | v0 | 2026-06-22 21:48:06 | WRITE


In [18]:
# ──────────────────────────────────────────────
# Finalização
# ──────────────────────────────────────────────
spark.stop()

failed = [e["table"] for e in write_log if e["status"] == "error"]
if failed:
    raise RuntimeError(f"Falha na escrita de {len(failed)} tabela(s): {failed}")

log.info("✔ Pipeline Silver → Gold finalizado com sucesso!")

2026-06-22 21:48:10  INFO      ✔ Pipeline Silver → Gold finalizado com sucesso!
